# Baseline models: regressione logistica e albero decisionale

Questo notebook implementa il punto 1 del progetto: due pipeline baseline per predire la sopravvivenza complessiva a 5 anni. Il preprocessing è incluso in ciascuna pipeline e viene appreso solo sui dati di training. Il test set resta riservato alla valutazione finale.


## 1. Import e configurazione


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
HORIZON = 60  # mesi: 5 anni
np.random.seed(SEED)
pd.set_option("display.max_columns", None)


## 2. Caricamento e definizione dell’esito

Il notebook cerca automaticamente il CSV nella cartella del progetto e nella cartella di lavoro. Se non lo trova, prova a montare Google Drive quando è eseguito in Colab. In locale, assicurati che il file CSV sia nella cartella del progetto o nella cartella da cui avvii Jupyter.


In [5]:
CSV_NAME = "Copia di SEER Breast Cancer Dataset .csv"
PROJECT_DIR = Path(r"C:\Progetti\neuroengineering_course\progetto")
LOCAL_CANDIDATES = [PROJECT_DIR / CSV_NAME, Path.cwd() / CSV_NAME]
DATASET_PATH = next((path for path in LOCAL_CANDIDATES if path.exists()), None)

if DATASET_PATH is None:
    try:
        from google.colab import drive
    except ImportError as exc:
        searched = ", ".join(str(path) for path in LOCAL_CANDIDATES)
        raise FileNotFoundError(
            f"CSV non trovato. Percorsi locali controllati: {searched}. "
            "Metti il file nella cartella del progetto o nella cartella di lavoro."
        ) from exc

    drive.mount("/content/drive")
    DATASET_PATH = Path("/content/drive/MyDrive/progettoneuro") / CSV_NAME

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"CSV non trovato in {DATASET_PATH}. Controlla il percorso nel Drive."
    )

print(f"Dataset caricato da: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
df = df.drop(columns=["Unnamed: 3"], errors="ignore")
df.columns = df.columns.str.strip()
df = df.drop_duplicates().reset_index(drop=True)
assert df.isna().sum().sum() == 0, "Il dataset contiene valori mancanti da gestire."

survived_5y = df[df["Survival Months"] >= HORIZON].copy()
survived_5y["label"] = 0
died_before_5y = df[
    (df["Status"] == "Dead") & (df["Survival Months"] < HORIZON)
].copy()
died_before_5y["label"] = 1
censored_before_5y = df[
    (df["Status"] == "Alive") & (df["Survival Months"] < HORIZON)
]
task_df = pd.concat([survived_5y, died_before_5y], ignore_index=True)

print(f"Pazienti valutabili: {len(task_df)}")
print(f"Censurate ed escluse: {len(censored_before_5y)}")
print("Distribuzione delle classi (proporzioni):")
print(task_df["label"].value_counts(normalize=True).sort_index().round(3))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset caricato da: /content/drive/MyDrive/progettoneuro/Copia di SEER Breast Cancer Dataset .csv
Pazienti valutabili: 3270
Censurate ed escluse: 753
Distribuzione delle classi (proporzioni):
label
0    0.863
1    0.137
Name: proportion, dtype: float64


## 3. Split stratificato

Le colonne usate per costruire l’esito non entrano tra le feature. La stratificazione mantiene proporzioni simili delle classi nel training e nel test.


In [6]:
LEAKAGE_COLS = ["Survival Months", "Status", "label"]
X = task_df.drop(columns=LEAKAGE_COLS)
y = task_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, " Test riservato:", X_test.shape)
print(
    "Prevalenza label=1 — train:", round(y_train.mean(), 3),
    " test:", round(y_test.mean(), 3)
)


Train: (2452, 13)  Test riservato: (818, 13)
Prevalenza label=1 — train: 0.137  test: 0.137


## 4. Preprocessing comune

Le variabili numeriche sono standardizzate e quelle categoriche codificate con one-hot encoding. T e N Stage hanno un ordine clinico; qui usiamo one-hot come scelta baseline, senza imporre distanze numeriche tra livelli. La codifica ordinale può essere confrontata in una successiva analisi.


In [7]:
NUM_COLS = [
    "Age", "Tumor Size", "Regional Node Examined", "Reginol Node Positive"
]
CAT_COLS = [
    "Race", "Marital Status", "T Stage", "N Stage", "6th Stage",
    "Grade", "A Stage", "Estrogen Status", "Progesterone Status"
]
assert set(NUM_COLS) | set(CAT_COLS) == set(X.columns)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])


## 5. Due pipeline baseline

Entrambi i modelli usano class_weight="balanced" per compensare lo sbilanciamento durante l’addestramento senza ricampionare i dati. Le impostazioni sono punti di partenza: il confronto delle prestazioni e il tuning degli iperparametri vengono dopo, con cross-validation sul solo training set.


In [8]:
logistic_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED
    )),
])

decision_tree_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", DecisionTreeClassifier(
        class_weight="balanced", random_state=SEED
    )),
])

baseline_models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
}
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    print(f"{name}: addestrato sul training set ({len(X_train)} pazienti)")


Logistic Regression: addestrato sul training set (2452 pazienti)
Decision Tree: addestrato sul training set (2452 pazienti)


## Prossimo passaggio

Confronta questi modelli sul training set con la stessa cross-validation stratificata e metriche adatte allo sbilanciamento (ad esempio recall della classe 1, F1, balanced accuracy e PR-AUC). Solo dopo la selezione finale valuta il modello una volta sul test set.


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "balanced_accuracy": "balanced_accuracy",
    "pr_auc": "average_precision",
}

results = []

for name, pipeline in baseline_models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
    )

    row = {"Modello": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric} media"] = values.mean()
        row[f"{metric} std"] = values.std()

    results.append(row)

results_df = pd.DataFrame(results).set_index("Modello")
display(results_df.round(3))

,precision media,precision std,recall media,recall std,f1 media,f1 std,balanced_accuracy media,balanced_accuracy std,pr_auc media,pr_auc std
Modello,,,,,,,,,,
Logistic Regression,0.285,0.031,0.671,0.048,0.400,0.038,0.700,0.034,0.385,0.050
Decision Tree,0.220,0.030,0.226,0.022,0.223,0.026,0.549,0.015,0.157,0.009
